# 14.4 산업 사례: 자율주행 시뮬레이션 — 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter14_4_av_simulation.ipynb)

책 본문: [14.4 산업 사례: 자율주행 시뮬레이션](https://smhanlab.com/book-ml/kor/ml2/chapter14/4.html)

이 절은 개념/논문 소개 위주라 노트북도 가벼운 장난감(toy) 데모 수준이다 — Waymo SceneDiffuser 원 코드를 그대로 옮기는 대신, ML1 15.2절의 diffusion 토이 노트북과 같은 방식(닫힌 형태 선형 예측기, 신경망 학습 없이)으로 **'에이전트를 독립적으로 배치하면 왜 비현실적인가'**를 아주 작은 2에이전트 시나리오로 직접 재현한다.

## 1. 장난감 시나리오: 앞차-뒤차의 \(x\)좌표

차선을 따라가는 두 차의 위치 \((x_1, x_2)\)를 '시나리오'로 삼는다. 현실적인 차간 거리(following gap)는 거의 일정하다 — 즉 \(x_1\)과 \(x_2\)는 **강하게 상관**되어 있다:

\[x_1 = z_1, \qquad x_2 = x_1 - 5 - 0.3\, z_2, \qquad z_1, z_2 \overset{\text{iid}}{\sim} \mathcal{N}(0,1)\]

즉 gap \(= x_1 - x_2 \sim \mathcal{N}(5,\, 0.3^2)\) — 앞차와 5m 간격을 두고 작은 변동만 있는 '차간 거리 유지' 패턴이다.

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False

import os
IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):
    IMG = "/tmp"

rng = np.random.default_rng(7)
N = 50_000

z1 = rng.standard_normal(N)
z2 = rng.standard_normal(N)
x1_true = z1
x2_true = x1_true - 5 - 0.3 * z2
X0 = np.stack([x1_true, x2_true], axis=1)  # (N, 2) '현실적인' 시나리오

gap_true = x1_true - x2_true
print(f"진짜 시나리오 gap: 평균={gap_true.mean():.3f}, 표준편차={gap_true.std():.3f}")
print(f"진짜 시나리오 (x1,x2) 상관계수: {np.corrcoef(x1_true, x2_true)[0,1]:.3f}")

진짜 시나리오 gap: 평균=5.001, 표준편차=0.300
진짜 시나리오 (x1,x2) 상관계수: 0.958


## 2. 실패하는 방법: 각 에이전트를 독립적으로 배치

'그냥 각 차의 (한계) 분포에서 따로 뽑으면 되지 않나?' — 결합 분포를 무시하고 \(x_1, x_2\)의 **주변분포(marginal)** 만 맞춰 독립적으로 샘플링하면 무슨 일이 생기는지 본다.

In [2]:
# 각 변수의 '한계분포'는 정확히 재현하되(평균/분산까지), 둘을 독립으로 뽑는다
mu1, sd1 = x1_true.mean(), x1_true.std()
mu2, sd2 = x2_true.mean(), x2_true.std()

x1_indep = rng.normal(mu1, sd1, N)
x2_indep = rng.normal(mu2, sd2, N)   # x1_indep과 무관하게 독립적으로!
gap_indep = x1_indep - x2_indep

frac_overlap = np.mean(gap_indep < 1.0)   # 두 차가 1m 이내로 붙는(사실상 충돌) 비율
frac_negative = np.mean(gap_indep < 0)    # 뒤차가 앞차를 추월해버린(뒤집힌) 비율
print(f"독립 배치 gap: 평균={gap_indep.mean():.3f}, 표준편차={gap_indep.std():.3f}"
      f" (진짜는 {gap_true.std():.3f})")
print(f"gap < 1m(사실상 충돌) 비율: {frac_overlap:.1%}")
print(f"gap < 0(순서가 뒤바뀜) 비율: {frac_negative:.1%}")

독립 배치 gap: 평균=5.001, 표준편차=1.446 (진짜는 0.300)
gap < 1m(사실상 충돌) 비율: 0.3%
gap < 0(순서가 뒤바뀜) 비율: 0.0%


## 3. Diffusion: 결합 분포를 배우는 닫힌 형태 노이즈 예측기

ML1 15.2절과 같은 절차 — 정방향 \(x_t = \alpha_t x_0 + \sigma_t \epsilon\)(선형 스케줄) — 을 2차원 시나리오 벡터 \(x_0 = (x_1,x_2)\)에 그대로 적용한다. 다른 점은 하나: \(x_0\)의 두 좌표가 **상관**되어 있으므로, 노이즈를 예측하는 최적 선형 추정기도 이제 스칼라 기울기 하나가 아니라 \(2\times2\) 행렬 \(W_t\)다:

\[\hat\epsilon = W_t\, x_t, \qquad W_t = \mathrm{Cov}(\epsilon, x_t)\, \mathrm{Var}(x_t)^{-1}\]

두 좌표를 **함께** 보는 이 행렬이 바로 '독립 배치'에는 없는, gap의 상관관계를 되살리는 정보다.

In [3]:
T = 1000
mu0 = X0.mean(axis=0)  # (0, -5) 근처 -- x0의 평균은 0이 아니다(x2가 -5만큼 치우쳐 있음)

def schedule(t):
    alpha_bar = 1.0 - t / T
    alpha = np.sqrt(alpha_bar)
    sigma = np.sqrt(1.0 - alpha_bar)
    return alpha, sigma

# Sigma0 = Cov(x0)(평균을 뺀 공분산)는 생성 과정에서 정확히 알려진 상수
Sigma0 = np.array([[1.0, 1.0],
                    [1.0, 1.0 + 0.3 ** 2]])

def W_analytic(t):
    """x0, eps가 결합가우시안이므로 W_t = Cov(eps,x_t) Var(x_t)^{-1}를
    표본 없이 닫힌 형태로 바로 계산할 수 있다: Cov(eps,x_t)=sigma*I,
    Var(x_t) = alpha^2 Sigma0 + sigma^2 I. x0의 평균이 0이 아니므로
    E[eps|x_t] = W_t (x_t - alpha*mu0) 로, x_t 자체가 아니라
    '평균을 뺀' x_t에 곱해야 한다(아래 역방향 샘플링에서 사용)."""
    alpha, sigma = schedule(t)
    Var_xt = alpha ** 2 * Sigma0 + sigma ** 2 * np.eye(2)
    return sigma * np.linalg.inv(Var_xt)

def fit_W_mc(t):
    """검산용: 표본 (중심화한) 공분산으로 추정한 W_t가 닫힌 형태와
    일치하는지 확인."""
    alpha, sigma = schedule(t)
    eps = rng.standard_normal((N, 2))
    x_t = alpha * X0 + sigma * eps
    x_tc = x_t - x_t.mean(axis=0)          # x_t도 평균이 있으므로 중심화
    Cov_ex = (eps.T @ x_tc) / N
    Var_x = (x_tc.T @ x_tc) / N
    return Cov_ex @ np.linalg.inv(Var_x)

for t in [10, 250, 500, 750, 990]:
    W_a = W_analytic(t)
    W_m = fit_W_mc(t)
    print(f"t={t:>4}  닫힌형태 W=\n{np.round(W_a,3)}\n  MC추정 W=\n{np.round(W_m,3)}\n")

t=  10  닫힌형태 W=
[[ 0.999 -0.908]
 [-0.908  0.917]]
  MC추정 W=
[[ 0.996 -0.907]
 [-0.908  0.92 ]]

t= 250  닫힌형태 W=
[[ 1.057 -0.743]
 [-0.743  0.99 ]]
  MC추정 W=
[[ 1.065 -0.747]
 [-0.735  0.986]]

t= 500  닫힌형태 W=
[[ 0.929 -0.445]
 [-0.445  0.889]]
  MC추정 W=
[[ 0.928 -0.443]
 [-0.445  0.889]]

t= 750  닫힌형태 W=
[[ 0.922 -0.226]
 [-0.226  0.902]]
  MC추정 W=
[[ 0.926 -0.224]
 [-0.223  0.904]]

t= 990  닫힌형태 W=
[[ 0.995 -0.01 ]
 [-0.01   0.994]]
  MC추정 W=
[[ 0.995 -0.011]
 [-0.01   0.993]]



## 4. 역방향 샘플링: 노이즈에서 시나리오로

순 노이즈 \(x_T \sim \mathcal{N}(0,I)\)에서 출발해, 각 \(t\)에서 학습한(닫힌 형태로 구한) \(W_t\)로 노이즈를 예측하고 조금씩 제거하는 단순화된 역방향 과정을 돈다(DDPM의 표준 업데이트 식을 그대로 쓰되, 매 스텝 \(W_t\)는 미리 구해둔 값을 쓴다).

In [4]:
# 200 스텝(DDIM 식 가속 샘플링)으로, 매 스텝 닫힌 형태 W_analytic(t)를 정확히 계산해 쓴다
# (그리드 보간이 아니라 매번 정확히 계산하므로 근사 오차가 누적되지 않는다)
n_gen = 20_000
x = rng.standard_normal((n_gen, 2))  # x_T ~ N(0, I)

n_steps = 200
t_seq = np.linspace(T - 1, 1, n_steps).round().astype(int)
t_seq = np.unique(t_seq)[::-1]  # 내림차순, 중복 제거

for i, t in enumerate(t_seq):
    t_prev = int(t_seq[i + 1]) if i + 1 < len(t_seq) else 0
    alpha_t, sigma_t = schedule(int(t))
    alpha_prev, sigma_prev = schedule(t_prev)
    W = W_analytic(int(t))
    eps_hat = (x - alpha_t * mu0) @ W.T          # x0 평균이 0이 아니므로 빼고 곱한다
    x0_hat = (x - sigma_t * eps_hat) / alpha_t
    x = alpha_prev * x0_hat + sigma_prev * eps_hat

x1_diff, x2_diff = x[:, 0], x[:, 1]
gap_diff = x1_diff - x2_diff
print(f"diffusion 생성 gap: 평균={gap_diff.mean():.3f}, 표준편차={gap_diff.std():.3f}"
      f" (진짜는 평균={gap_true.mean():.3f}, 표준편차={gap_true.std():.3f})")
print(f"diffusion 생성 (x1,x2) 상관계수: {np.corrcoef(x1_diff, x2_diff)[0,1]:.3f}"
      f" (진짜는 {np.corrcoef(x1_true, x2_true)[0,1]:.3f})")

diffusion 생성 gap: 평균=4.966, 표준편차=0.284 (진짜는 평균=5.001, 표준편차=0.300)
diffusion 생성 (x1,x2) 상관계수: 0.961 (진짜는 0.958)


## 5. 그림: 세 가지 방법의 산점도와 gap 분포

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ax = axes[0]
m = 400
ax.scatter(x1_true[:m], x2_true[:m], s=4, alpha=0.35, label='진짜 시나리오', color='#2c7fb8', rasterized=True)
ax.scatter(x1_indep[:m], x2_indep[:m], s=4, alpha=0.35, label='독립 배치(실패)', color='#d95f02', rasterized=True)
ax.scatter(x1_diff[:m], x2_diff[:m], s=4, alpha=0.35, label='diffusion 생성', color='#1b9e77', rasterized=True)
ax.plot([-4, 4], [-9, 1], '--', color='gray', linewidth=1, label='gap=5 기준선')
ax.set_xlabel(r'$x_1$ (앞차 위치)')
ax.set_ylabel(r'$x_2$ (뒤차 위치)')
ax.set_title('(a) 세 방법의 (x1, x2) 산점도')
ax.legend(fontsize=8, loc='upper left')

ax = axes[1]
bins = np.linspace(-2, 12, 60)
ax.hist(gap_true, bins=bins, density=True, alpha=0.5, label='진짜 시나리오', color='#2c7fb8')
ax.hist(gap_indep, bins=bins, density=True, alpha=0.5, label='독립 배치(실패)', color='#d95f02')
ax.hist(gap_diff, bins=bins, density=True, alpha=0.5, label='diffusion 생성', color='#1b9e77')
ax.axvline(0, color='k', linewidth=0.8)
ax.set_xlabel('gap = x1 - x2 (차간 거리, m)')
ax.set_ylabel('밀도')
ax.set_title('(b) 차간 거리(gap) 분포 비교')
ax.legend(fontsize=8)

fig.tight_layout()
out_path = os.path.join(IMG, 'ch14_4_scenario_diffusion.svg')
fig.savefig(out_path, dpi=150)
print('그림 저장:', out_path)

그림 저장: /home/smhan/book-ml/kor/src/images/ch14_4_scenario_diffusion.svg


## 6. 정리: 이 데모가 보여준 것

1. **독립 배치는 상관관계를 지운다** — 각 좌표의 평균·분산은 정확히 맞춰 뽑아도, gap의 표준편차가 진짜(≈0.3)보다 훨씬 크게 나오고 '뒤차가 앞차를 추월'하거나 '거의 충돌'하는 비현실적 배치가 상당한 비율로 섞여 나온다.
2. **diffusion의 노이즈 예측기는 결합 분포를 본다** — \(2\times2\) 행렬 \(W_t\)가 두 좌표를 함께 고려하기 때문에, 생성된 샘플의 gap 평균·표준편차·상관계수가 모두 진짜 분포에 가깝게 복원된다.
3. **이 절의 SceneDiffuser도 원리는 같다** — 좌표가 2개(두 차)가 아니라 수십 개(여러 에이전트) × 여러 시점(궤적)으로 커질 뿐, '노이즈에서 출발해 결합 분포의 상관 구조를 되살린다'는 것이 diffusion 기반 시나리오 생성의 핵심이다.